# Optimizar código — medir antes de optimizar

**Unidad 4.c** · Acompaña a `Clase_04_DatosPanel` · Notas: cap. 5

Tres implementaciones de la **misma** transformación intragrupos, de la más ingenua a la
vectorizada. Las tres dan el mismo resultado —lo verificamos— y difieren en tiempo por
dos órdenes de magnitud.

**Antes de ejecutar:** predice cuál será más rápida y por cuánto. Anótalo.

In [ ]:
import time

import numpy as np
import pandas as pd

panel = pd.read_csv("../../Clase_04_DatosPanel/wage_panel.csv")
VARIABLE = "lwage"
GRUPO = "nr"

print(f"Observaciones: {len(panel)}   Grupos: {panel[GRUPO].nunique()}")

In [ ]:
def version_1_ciclo(panel):
    """Ciclo explícito sobre los grupos, con concatenación al final."""
    partes = []
    for _, bloque in panel.groupby(GRUPO):
        partes.append(bloque[VARIABLE] - bloque[VARIABLE].mean())
    return pd.concat(partes).sort_index()


def version_2_apply(panel):
    """`apply` sobre el objeto agrupado: más corto, todavía un ciclo por dentro."""
    return (
        panel.groupby(GRUPO)[VARIABLE]
        .apply(lambda s: s - s.mean())
        .reset_index(level=0, drop=True)
        .sort_index()
    )


def version_3_transform(panel):
    """Vectorizada: `transform` difunde la media sin ciclo en Python."""
    return panel[VARIABLE] - panel.groupby(GRUPO)[VARIABLE].transform("mean")

## Primero: ¿dan el mismo resultado?

Éste es el orden correcto. **Una optimización que cambia el resultado no es una
optimización**, y comparar tiempos de cosas distintas no significa nada.

In [ ]:
resultados = {
    "1. ciclo explícito": version_1_ciclo(panel),
    "2. groupby + apply": version_2_apply(panel),
    "3. groupby + transform": version_3_transform(panel),
}

base = resultados["3. groupby + transform"]
for nombre, salida in resultados.items():
    maxima = np.abs(salida.values - base.values).max()
    iguales = np.allclose(salida.values, base.values)
    print(f"  {nombre:24s} {'idéntico' if iguales else 'DIFIERE'}   (máx. dif. {maxima:.2e})")

## Y la propiedad matemática, que no depende de la implementación

Una variable centrada por grupo tiene media cero dentro de cada grupo.

In [ ]:
medias = pd.DataFrame({GRUPO: panel[GRUPO], "c": base}).groupby(GRUPO)["c"].mean()
print(f"Media intragrupo máxima en valor absoluto: {medias.abs().max():.2e}")
print("(debe ser cero hasta precisión de punto flotante)")

## Ahora sí: los tiempos

In [ ]:
def cronometrar(funcion, panel, repeticiones=5):
    """Devuelve el mejor tiempo de varias corridas, en milisegundos."""
    tiempos = []
    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion(panel)
        tiempos.append(time.perf_counter() - inicio)
    return min(tiempos) * 1000


tiempos = {
    "1. ciclo explícito": cronometrar(version_1_ciclo, panel),
    "2. groupby + apply": cronometrar(version_2_apply, panel),
    "3. groupby + transform": cronometrar(version_3_transform, panel),
}

referencia = min(tiempos.values())
for nombre, ms in tiempos.items():
    print(f"  {nombre:24s} {ms:8.2f} ms   ({ms / referencia:5.1f}x)")

## La pregunta que importa

La versión más lenta tarda unas decenas de milisegundos sobre 4,360 observaciones.

**¿Valía la pena optimizarla?**

Para este panel, **no**: la diferencia es imperceptible para quien corre el cuaderno una
vez, y el tiempo del alumno que lo lee vale más que el del procesador. Sí valdría con un
panel mil veces más grande, o dentro de un *bootstrap* de 10,000 réplicas, donde los
milisegundos se vuelven horas.

La razón real para preferir la versión 3 **no es la velocidad**: es que es la que menos
ocasión da de equivocarse. Compárala con el
[BUG 04](../02_Depurar/Bug_04_Panel_Desalineado.ipynb), donde la variante «obvia» de esa
misma línea destruye el 94 % de la muestra sin avisar.

> **Optimizar sin medir no es ingeniería, es superstición.** Y medir a veces dice que no
> había nada que hacer.

## Tareas

1. Pídele a un asistente de IA que optimice la versión 1. ¿Llegó a la versión 3, a la 2,
   o a otra cosa? ¿Introdujo algún cambio que altere el resultado?
2. Repite la medición con un panel artificialmente replicado 100 veces
   (`pd.concat([panel] * 100)`). ¿Cambian las proporciones? ¿Cambia tu respuesta a «¿valía
   la pena?»?
3. ¿En qué punto del curso te encontrarás con un cálculo donde sí importe? *Pista:*
   `05_Reproducibilidad` hace 2,000 réplicas de un *bootstrap*.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Ver el [README de la carpeta](README.md) para las demás actividades.

In [ ]:
# Tu experimento aquí.